In [28]:
import math

In [29]:
class Value:

    def __init__(self, data, prev = None):
        self.data = data
        self.prev = prev if prev is not None else ()
        self.grad = 0
        # why need this now 
        #we need to send our grad (calculate and send)
        #to our ancestors this function takes our grad and calulate their grad and accumulates
        #the grad with respect to loss through this node
        #when our descendants call backward they asssign us grad
        #we call grad to assign our ancestors grad (assign == accumulate)
        #_backward i know how to contribute to descendants grads using my grad
        self._backward = lambda : None

    def __repr__(self):
        return (f"Value(data={self.data},grad={self.grad})")
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, prev =(self, other))
        def backward():
            self.grad +=  out.grad
            other.grad += out.grad
        out._backward = backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, prev = (self, other))
        def backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = backward
        return out


    def __radd__(self, other):
        return self.__add__(other)

    def __rmul__(self, other):
        return self.__mul__(other)

    def __neg__(self):
        return self.__mul__(-1.0)


    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, prev = (self,))
        def backward():
            self.grad += (other * (self.data ** (other -1))) * out.grad
        out._backward = backward
        return out

    def __sub__(self, other):
        return self.__add__(-other)


    def __truediv__(self, other):
        #other could be value type or int/float type
        #so don't directly call other__pow__()
        #instead use operator ** which invokes
        #whatever needed accordingly
        return self.__mul__(other ** -1)


    def __rtruediv__(self, other):
        #other might be a scalar (int, float) sclar/Value calls this so
        return Value(other) / self


    def __rsub__(self, other):
        return Value(other) - self

    def exp(self):
        out = Value(math.exp(self.data), prev = (self,))
        def backward():
            self.grad += out.data * out.grad 
        out._backward = backward
        return out


    def sigmoid(self):
        out = 1/(1+(self.exp() ** -1))
        return out

# x -> e^x -> 1/1+y^-1

    def backward(self):
        nodes = []
        visited = set()
        def visit(node):
            if node not in visited:
                visited.add(node)
                for child in node.prev:
                        visit(child)
                nodes.append(node)

        visit(self)
        for node in nodes:
            node.grad = 0

        self.grad = 1
        for node in reversed(nodes):
            node._backward()
        print(nodes)

In [30]:
x = Value(2)
w = Value(3)
b = Value(5)
xw = x * w
y = xw + b
l = y - 12

In [31]:
l.backward()

[Value(data=2,grad=3), Value(data=3,grad=2), Value(data=6,grad=1), Value(data=5,grad=1), Value(data=11,grad=1), Value(data=-12,grad=1), Value(data=-1,grad=1)]


In [32]:
l.grad  = 1
p = [l, y, xw, b, w, x]

# for i in p:
#     i._backward()
#     print(i)

In [33]:
v1  = Value(3)
v2 = v1.sigmoid()
v2.backward()


[Value(data=1,grad=0.9525741268224334), Value(data=3,grad=0.04517665973091214), Value(data=20.085536923187668,grad=0.0022492134466546485), Value(data=0.049787068367863944,grad=-0.9073974670915212), Value(data=1,grad=-0.9073974670915212), Value(data=1.0497870683678638,grad=-0.9073974670915212), Value(data=0.9525741268224334,grad=1), Value(data=0.9525741268224334,grad=1)]
